In [1]:
import os
import pandas as pd
import numpy as np
import json


In [4]:
import json
from collections import Counter
import os

file_path = os.path.join(
    "C:\\", "Users", "thiago-ext", "Documents", "FNIRS", "psychopy", "filtered_questions.json"
)

def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return data

data = load_json(file_path)

# If the JSON is a list of dicts, each with a "type" field:
type_counts = Counter(item.get("type", "MISSING_TYPE") for item in data)

print(type_counts)



Counter({'abstract': 22, 'concrete': 21})


In [5]:
import json, os, re, shutil, datetime
from copy import deepcopy

# ====== CONFIG ======
# Edit this path if needed:
file_path = r"C:\Users\thiago-ext\Documents\FNIRS\psychopy\filtered_questions.json"

# JSON item fields we may edit
TEXT_FIELDS = [
    "question_text_translated",
    "question_itself_translated",
    "question_option_A_translated",
    "question_option_B_translated",
    "question_option_C_translated",
    "question_option_D_translated",
    "question_option_E_translated",
]

# ---------- helpers ----------
def wb_replace(text, src, dst, flags=0):
    """Word-boundary sensitive replacement (case-sensitive by default)."""
    pattern = r"\b" + re.escape(src) + r"\b"
    return re.sub(pattern, dst, text, flags=flags)

def replace_phrase(text, pairs):
    """Apply a list of (pattern, repl, is_regex) sequentially."""
    out = text
    for pat, repl, is_regex in pairs:
        if out is None:
            break
        out = re.sub(pat, repl, out) if is_regex else out.replace(pat, repl)
    return out

def edit_field(d, field, fn):
    """Apply a callable to a field if present and non-empty."""
    if field in d and isinstance(d[field], str) and d[field].strip():
        new_val = fn(d[field])
        return new_val if new_val != d[field] else None
    return None

def note_rewrite(item, msg):
    item["needs_substantive_rewrite"] = True
    item["editor_note"] = msg

def show_change(qid, field, before, after):
    print(f"[Q{qid}] {field}:")
    print("  - BEFORE:", before)
    print("  - AFTER :", after)
    print()

def apply_change(item, field, new_val, changes):
    if new_val is not None:
        changes.append((field, item[field], new_val))
        item[field] = new_val

# ---------- load & backup ----------
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = file_path + f".bak_{ts}"
shutil.copy2(file_path, backup_path)
print(f"Backup created at: {backup_path}\n")

total_changes = 0

# ---------- per-question edits ----------
for item in data:
    if not isinstance(item, dict):
        continue
    qn = item.get("question_number")
    changes = []

    # Q20: "deal" -> "deals"; pronoun consistency ("your" -> "his") in options
    if qn == 20:
        # In text/stem: deal -> deals
        for f in ["question_text_translated", "question_itself_translated"]:
            new_val = edit_field(item, f, lambda t: wb_replace(t, "deal", "deals"))
            apply_change(item, f, new_val, changes)
        # In options: standardize "your" to "his" (minimal, targeted to Q20 only)
        for f in ["question_option_A_translated","question_option_B_translated",
                  "question_option_C_translated","question_option_D_translated",
                  "question_option_E_translated"]:
            def pronoun_fix(t):
                t2 = re.sub(r"\byour\b", "his", t)
                t2 = re.sub(r"\bYour\b", "His", t2)
                return t2
            new_val = edit_field(item, f, pronoun_fix)
            apply_change(item, f, new_val, changes)

    # Q26: "This lyric" -> "These lyrics"; match has/have form
    if qn == 26:
        def fix_lyric(t):
            t = re.sub(r"\bThis lyric has\b", "These lyrics have", t)
            t = re.sub(r"\bThis lyric\b", "These lyrics", t)
            return t
        for f in ["question_text_translated", "question_itself_translated"]:
            new_val = edit_field(item, f, fix_lyric)
            apply_change(item, f, new_val, changes)

    # Q33: needs substantive rewrite (do not alter text)
    if qn == 33:
        note_rewrite(item, "Q33: 'Traits of a survivor:' is ambiguous/awkward; rewrite needed beyond grammar.")

    # Q46: light grammar so stem matches options (result in -> noun phrase)
    if qn == 46:
        def fix_stem(t):
            # minimal: cause -> result in the individual's ...
            t = t.replace("cause the individual to", "result in the individual's")
            return t
        new_val = edit_field(item, "question_itself_translated", fix_stem)
        apply_change(item, "question_itself_translated", new_val, changes)

    # Q48: ensure article; if stem ends with "is" add " a/an"
    if qn == 48:
        def ensure_article(t):
            # if stem ends with ' is' or ' is a', keep minimal fix
            if re.search(r"\bis\s*$", t):
                return t + " a"
            return t
        new_val = edit_field(item, "question_itself_translated", ensure_article)
        apply_change(item, "question_itself_translated", new_val, changes)

    # Q49: translation strange -> flag
    if qn == 49:
        note_rewrite(item, "Q49: Replace Merleau-Ponty passage with an official English translation.")

    # Q51: confusing/repetitive sentence -> flag
    if qn == 51:
        note_rewrite(item, "Q51: Confusing sentence about 'network' repeated; rewrite for clarity.")

    # Q56: first sentence unclear -> flag
    if qn == 56:
        note_rewrite(item, "Q56: First sentence unclear; rewrite for clarity.")

    # Q59: relies on external knowledge -> flag (validity check)
    if qn == 59:
        note_rewrite(item, "Q59: Ensure the answer is inferable only from the text (avoid external knowledge).")

    # Q64: "favors the occupation..." -> "influences the occupation..."
    if qn == 64:
        def fix_phrase(t):
            return t.replace("favors the occupation of geographic space",
                             "influences the occupation of geographic space")
        new_val = edit_field(item, "question_itself_translated", fix_phrase)
        apply_change(item, "question_itself_translated", new_val, changes)

    # Q65: stem wording; pronouns -> they/them
    if qn == 65:
        def fix_stem(t):
            t = t.replace("expressed in the distinction between", "defined by the distinction between")
            t = t.replace("characterized by the distinction between", "defined by the distinction between")
            return t
        new_val = edit_field(item, "question_itself_translated", fix_stem)
        apply_change(item, "question_itself_translated", new_val, changes)

        # pronoun normalization in text/stem/options: him/her -> them/their
        def neutralize_pronouns(t):
            t = re.sub(r"\bhim\b", "them", t)
            t = re.sub(r"\bher\b", "them", t)
            t = re.sub(r"\bhis\b", "their", t)
            t = re.sub(r"\bher\b", "their", t)
            t = re.sub(r"\bHe\b", "They", t)
            t = re.sub(r"\bShe\b", "They", t)
            t = re.sub(r"\bhe\b", "they", t)
            t = re.sub(r"\bshe\b", "they", t)
            return t
        for f in TEXT_FIELDS:
            new_val = edit_field(item, f, neutralize_pronouns)
            apply_change(item, f, new_val, changes)

    # Q67: formatting note only -> flag
    if qn == 67:
        note_rewrite(item, "Q67: Use (a), (b), (c) instead of Roman numerals for multi-part items.")

    # Q68: "aciculifoliada" -> "aciculifoliate"
    if qn == 68:
        for f in TEXT_FIELDS:
            new_val = edit_field(item, f, lambda t: wb_replace(t, "Aciculifoliada", "Aciculifoliate"))
            if new_val is None:
                new_val = edit_field(item, f, lambda t: wb_replace(t, "aciculifoliada", "aciculifoliate"))
            apply_change(item, f, new_val, changes)

    # Q69: "derivatives" -> "derivative" (singular) in option with "import zone(s)"
    if qn == 69:
        for f in ["question_option_A_translated","question_option_B_translated",
                  "question_option_C_translated","question_option_D_translated",
                  "question_option_E_translated"]:
            def fix_derivative(t):
                # only if phrase resembles "... derivatives import ..."
                return re.sub(r"\bderivatives(\s+import)", r"derivative\1", t)
            new_val = edit_field(item, f, fix_derivative)
            apply_change(item, f, new_val, changes)

    # Q72: confusing/long -> flag
    if qn == 72:
        note_rewrite(item, "Q72: Simplify phrasing ('Breaking with the idea...'); restructure for clarity.")

    # Q75: misleading + repetition; switch to (a,b,c) formatting -> flag
    if qn == 75:
        note_rewrite(item, "Q75: Review for potential misleading cues and repetitive phrasing; use (a),(b),(c) for multi-part.")

    # Q77: requires definition of peneplain; also fix common typo 'paneplain' -> 'peneplain'
    if qn == 77:
        for f in TEXT_FIELDS:
            new_val = edit_field(item, f, lambda t: wb_replace(t, "paneplain", "peneplain"))
            apply_change(item, f, new_val, changes)
        note_rewrite(item, "Q77: Define 'peneplain' in-text or rephrase so answer is inferable without outside knowledge.")

    # Q78: needs more context (Kaizen/Toyotism) -> flag
    if qn == 78:
        note_rewrite(item, "Q78: Add minimal context for Kaizen/Toyotism; current clue too vague.")

    # Q82: options wording fixes
    if qn == 82:
        def fix_A(t):  # only fertile -> only fertile soils
            return re.sub(r"\bonly fertile\b", "only fertile soils", t)
        def fix_B(t):  # steep slope -> steep slopes
            return re.sub(r"\bsteep slope\b", "steep slopes", t)
        new_val = edit_field(item, "question_option_A_translated", fix_A)
        apply_change(item, "question_option_A_translated", new_val, changes)
        new_val = edit_field(item, "question_option_B_translated", fix_B)
        apply_change(item, "question_option_B_translated", new_val, changes)
        note_rewrite(item, "Q82: Ensure the question is answerable from the provided text; align with 'contrary to what would be sensible' phrasing.")

    # Q83: 'testos' -> 'texts'
    if qn == 83:
        for f in TEXT_FIELDS:
            new_val = edit_field(item, f, lambda t: wb_replace(t, "testos", "texts"))
            apply_change(item, f, new_val, changes)

    # Q86: "has as goal" -> "has as its goal"
    if qn == 86:
        def fix_goal(t):
            t = re.sub(r"\bhas as goal\b", "has as its goal", t)
            t = re.sub(r"\bhas as a goal\b", "has as its goal", t)
            return t
        new_val = edit_field(item, "question_itself_translated", fix_goal)
        apply_change(item, "question_itself_translated", new_val, changes)
        note_rewrite(item, "Q86: Review technical terminology in options for correctness and clarity.")

    # ----- report changes for this item -----
    for f, before, after in changes:
        total_changes += 1
        show_change(qn, f, before, after)

# ---------- save ----------
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Done. Total fields changed: {total_changes}")
print(f"File updated: {file_path}")


Backup created at: C:\Users\thiago-ext\Documents\FNIRS\psychopy\filtered_questions.json.bak_20251103_134448

[Q20] question_text_translated:
  - BEFORE: Why the stage entrepreneurship industry
will destroy you
If, in the past, huge books with seven hundred pages spit out formulas, equations and calculations that taught you how to deal with your company's cash flow, today they say: “You will get there! Believe me, you will win!”
Mindset, empowerment, millennials, networking, coworking, deal, business, deadline, salesman with a hunter profile… all of this is part of his vocabulary. The package of books is always identical and the experiences are conveyed in the same way: you are a single centimeter away from victory. Do not stop!
  - AFTER : Why the stage entrepreneurship industry
will destroy you
If, in the past, huge books with seven hundred pages spit out formulas, equations and calculations that taught you how to deals with your company's cash flow, today they say: “You will get ther

In [6]:
import json

file_path = r"C:\Users\thiago-ext\Documents\FNIRS\psychopy\filtered_questions_revised.json"

# All targets to remove as (year, field, question_number)
targets = {
    # earlier set
    ("2022", "CH", 64),
    ("2018", "LC", 33),
    ("2020", "LC", 20),
    ("2017", "CH", 86),
    ("2017", "CH", 72),
    ("2017", "CH", 53),
    # new set
    ("2019", "CH", 67),
    ("2019", "CH", 75),
    ("2018", "CH", 49),
    ("2018", "CH", 52),
    ("2018", "CH", 72),
    ("2017", "CH", 48),
    ("2017", "CH", 64),
}

def normalize_tuple(item):
    year = str(item.get("year", "")).strip()
    field = str(item.get("field", "")).strip()
    qnum = item.get("question_number", None)
    try:
        qnum = int(qnum)
    except (TypeError, ValueError):
        pass
    return (year, field, qnum)

# Load
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Filter
kept, removed = [], []
for itm in data:
    key = normalize_tuple(itm)
    if key in targets:
        removed.append(key)
    else:
        kept.append(itm)

# Overwrite (no backup)
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(kept, f, ensure_ascii=False, indent=2)

print(f"Removed {len(removed)} questions:")
for y, fld, qn in removed:
    print(f"  {y}\t{fld}_{qn}")
print(f"Remaining items: {len(kept)}")


Removed 13 questions:
  2018	LC_33
  2020	LC_20
  2017	CH_48
  2017	CH_53
  2017	CH_64
  2017	CH_72
  2017	CH_86
  2018	CH_49
  2018	CH_52
  2018	CH_72
  2019	CH_67
  2019	CH_75
  2022	CH_64
Remaining items: 30


In [7]:
import json, re

# ====== CONFIG ======
file_path = r"C:\Users\thiago-ext\Documents\FNIRS\psychopy\filtered_questions_revised.json"

# Fields that may contain text to fix
TEXT_FIELDS = [
    "question_text_translated",
    "question_itself_translated",
    "question_option_A_translated",
    "question_option_B_translated",
    "question_option_C_translated",
    "question_option_D_translated",
    "question_option_E_translated",
]

def apply(item, field, fn, changes):
    """Apply fn(string)->string to item[field] if present; record change."""
    if field in item and isinstance(item[field], str):
        before = item[field]
        after = fn(before)
        if after != before:
            item[field] = after
            changes.append((field, before, after))

def sub(old, new, flags=0):
    """Return a replacer fn using regex sub."""
    def _f(s): return re.sub(old, new, s, flags=flags)
    return _f

def replace(old, new):
    """Return a simple str.replace fn."""
    def _f(s): return s.replace(old, new)
    return _f

def wb_replace(word, repl, flags=0):
    """Word-boundary replacement (regex)."""
    pat = r"\b" + re.escape(word) + r"\b"
    return sub(pat, repl, flags=flags)

def fix_item(item):
    y = str(item.get("year", "")).strip()
    f = str(item.get("field", "")).strip()
    qn = int(item.get("question_number", -1))
    key = (y, f, qn)

    changes = []

    # -------- 2018 · LC · Q44 --------
    if key == ("2018", "LC", 44):
        apply(item, "question_text_translated", replace("copying In", "copying. In"), changes)
        apply(item, "question_text_translated", replace("being full, partial or paraphrases", "being full or partial, or paraphrases"), changes)
        apply(item, "question_text_translated", replace("searches section by section on search engines", "searches section by section in search engines"), changes)
        apply(item, "question_text_translated", replace("much more results", "many more results"), changes)

    # -------- 2020 · LC · Q31 --------
    if key == ("2020", "LC", 31):
        apply(item, "question_text_translated", wb_replace("postmenstrual", "postmenopausal", flags=re.IGNORECASE), changes)

    # -------- 2021 · LC · Q26 --------
    if key == ("2021", "LC", 26):
        apply(item, "question_text_translated", replace("This lyric has", "These lyrics have"), changes)
        apply(item, "question_text_translated", wb_replace("larger roundels", "heptasyllabic verse"), changes)

    # -------- 2017 · CH · Q47 --------
    # (no mandatory edits)

    # -------- 2017 · CH · Q68 --------
    if key == ("2017", "CH", 68):
        for fld in TEXT_FIELDS:
            apply(item, fld, wb_replace("aciculifoliada", "aciculifoliate"), changes)
            apply(item, fld, wb_replace("Aciculifoliada", "Aciculifoliate"), changes)

    # -------- 2017 · CH · Q69 --------
    if key == ("2017", "CH", 69):
        apply(item, "question_option_B_translated", sub(r"\bderivatives(\s+import)", r"derivative\1"), changes)
        # add missing periods to D/E
        apply(item, "question_option_D_translated", lambda s: s if s.strip().endswith(".") else s.rstrip() + ".", changes)
        apply(item, "question_option_E_translated", lambda s: s if s.strip().endswith(".") else s.rstrip() + ".", changes)

    # -------- 2017 · CH · Q75 --------
    if key == ("2017", "CH", 75):
        apply(item, "question_text_translated", replace("observer state not a member", "non-member observer state"), changes)

    # -------- 2018 · CH · Q77 --------
    if key == ("2018", "CH", 77):
        apply(item, "question_text_translated", replace("accumulations\nsurfaces", "surface accumulations"), changes)
        apply(item, "question_text_translated", replace("pebbles and the widespread", "pebbles, and the widespread"), changes)
        apply(item, "question_text_translated", replace("Accessed on: 8 July. 2015", "Accessed on: 8 July 2015"), changes)

    # -------- 2019 · CH · Q56 --------
    if key == ("2019", "CH", 56):
        apply(item, "question_itself_translated", replace("described exposes", "described expose"), changes)

    # -------- 2019 · CH · Q64 --------
    # (no mandatory edit)

    # -------- 2019 · CH · Q65 --------
    if key == ("2019", "CH", 65):
        apply(item, "question_itself_translated", replace("expressed in the distinction between", "defined by the distinction between"), changes)

    # -------- 2020 · CH · Q56 --------
    # (no edits)

    # -------- 2020 · CH · Q59 --------
    # (no edits)

    # -------- 2020 · CH · Q70 --------
    if key == ("2020", "CH", 70):
        apply(item, "question_itself_translated", replace("dated 1st BC", "dated 1st century BC"), changes)
        apply(item, "question_option_A_translated", lambda s: s if s.strip().endswith(".") else s.rstrip() + ".", changes)

    # -------- 2020 · CH · Q76 --------
    if key == ("2020", "CH", 76):
        # Clean the weird citation line
        apply(item, "question_text_translated",
              lambda s: s.replace("Belo Horizonte: table; São Paulo: Edusp 1977.",
                                  "Belo Horizonte; São Paulo: Edusp, 1977."),
              changes)

    # -------- 2020 · CH · Q77 --------
    # (no edits)

    # -------- 2020 · CH · Q78 --------
    # (no edits)

    # -------- 2020 · CH · Q82 --------
    if key == ("2020", "CH", 82):
        apply(item, "question_itself_translated", replace("the man's settlement", "human settlement"), changes)
        apply(item, "question_option_A_translated", wb_replace("only fertile", "only fertile soils"), changes)
        apply(item, "question_option_B_translated", wb_replace("steep slope", "steep slopes"), changes)

    # -------- 2020 · CH · Q84 --------
    if key == ("2020", "CH", 84):
        apply(item, "question_text_translated",
              replace("of both pre-existing rocks. as part of the rocks formed",
                      "of both pre-existing rocks and part of the rocks formed"),
              changes)

    # -------- 2021 · CH · Q46 --------
    if key == ("2021", "CH", 46):
        apply(item, "question_itself_translated",
              replace("relationships that cause the individual to", "relationships that result in the individual's"),
              changes)
        apply(item, "question_option_C_translated",
              replace("elevation of bureaucratic slaps", "increase in bureaucratic steps"),
              changes)

    # -------- 2021 · CH · Q51 --------
    if key == ("2021", "CH", 51):
        apply(item, "question_text_translated",
              replace("large-scale irrigation , as well as agricultural activities in\nsmall scale",
                      "large-scale irrigation, as well as agricultural activities on a small scale"),
              changes)

    # -------- 2021 · CH · Q57 --------
    if key == ("2021", "CH", 57):
        apply(item, "question_text_translated",
              replace("house on landings", "walk-up apartments"),
              changes)

    # -------- 2021 · CH · Q65 --------
    if key == ("2021", "CH", 65):
        # neutralize third-person pronouns while keeping second-person "your"
        def neutralize(s):
            s = re.sub(r"\bfrom him\b", "from them", s)
            s = re.sub(r"\bHer interests\b", "Their interests", s)
            s = re.sub(r"\bshe feels\b", "they feel", s)
            s = re.sub(r"\bher actions\b", "their actions", s)
            return s
        apply(item, "question_text_translated", neutralize, changes)
        # clean "idea of  surplus value" odd space
        apply(item, "question_option_B_translated", sub(r"\bidea of\s+surplus value\b", "idea of surplus value"), changes)

    # -------- 2021 · CH · Q76 --------
    if key == ("2021", "CH", 76):
        apply(item, "question_text_translated", replace("Latin .", "Latin."), changes)
        apply(item, "question_text_translated",
              replace("The spirituality of the Western Middle Ages, 19th century. VIII-XIII.",
                      "The spirituality of the Western Middle Ages, 8th–13th centuries."),
              changes)

    # -------- 2021 · CH · Q83 --------
    if key == ("2021", "CH", 83):
        apply(item, "question_text_translated", replace("above all, ,", "above all, "), changes)
        # replace 'testos' with 'pot lids' (and remove duplicate wording)
        apply(item, "question_text_translated", sub(r"carved\s+testos\s+or\s+pot\s+lids", "carved pot lids"), changes)

    # -------- 2021 · CH · Q86 --------
    if key == ("2021", "CH", 86):
        apply(item, "question_itself_translated", sub(r"has as\s*\n\s*goal", "has as its goal"), changes)

    # -------- 2022 · CH · Q47 --------
    # (no edits)

    # -------- 2022 · CH · Q51 --------
    if key == ("2022", "CH", 51):
        apply(item, "question_text_translated",
              replace("It is a network because it is made on a network\nglobal interaction between business networks.",
                      "It is a network because it is based on a global network of interactions between business networks."),
              changes)

    # -------- 2022 · CH · Q60 --------
    # (no edits)

    # -------- 2022 · CH · Q71 --------
    # (no edits)

    return changes

# ====== RUN ======
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

total_fields = 0
edited_questions = 0

for item in data:
    ch = fix_item(item)
    if ch:
        edited_questions += 1
        for field, before, after in ch:
            total_fields += 1
            print(f"[{item.get('year')}] {item.get('field')}_{item.get('question_number')} :: {field}")
            # Show a short diff-like preview
            before_snip = (before[:120] + "…") if len(before) > 120 else before
            after_snip  = (after[:120] + "…") if len(after) > 120 else after
            print("  - BEFORE:", before_snip)
            print("  - AFTER :", after_snip)
            print()

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Done. Edited {total_fields} field(s) across {edited_questions} question(s). File overwritten:\n{file_path}")


[2018] LC_44 :: question_text_translated
  - BEFORE: Plagiarism Sniffer: a tool against illegal copying In the academic world or in the media, illegal copies can appear in d…
  - AFTER : Plagiarism Sniffer: a tool against illegal copying. In the academic world or in the media, illegal copies can appear in …

[2018] LC_44 :: question_text_translated
  - BEFORE: Plagiarism Sniffer: a tool against illegal copying. In the academic world or in the media, illegal copies can appear in …
  - AFTER : Plagiarism Sniffer: a tool against illegal copying. In the academic world or in the media, illegal copies can appear in …

[2018] LC_44 :: question_text_translated
  - BEFORE: Plagiarism Sniffer: a tool against illegal copying. In the academic world or in the media, illegal copies can appear in …
  - AFTER : Plagiarism Sniffer: a tool against illegal copying. In the academic world or in the media, illegal copies can appear in …

[2018] LC_44 :: question_text_translated
  - BEFORE: Plagiarism Sniffe

In [8]:
import json, csv, os

# ====== CONFIG ======
file_path = r"C:\Users\thiago-ext\Documents\FNIRS\psychopy\filtered_questions_revised.json"
out_dir = os.path.dirname(file_path)
per_field_csv = os.path.join(out_dir, "question_letter_counts_per_field.csv")
per_question_csv = os.path.join(out_dir, "question_letter_counts_totals.csv")

# Text fields to analyze (add/remove if needed)
TEXT_FIELDS = [
    "question_text_translated",
    "question_itself_translated",
    "question_option_A_translated",
    "question_option_B_translated",
    "question_option_C_translated",
    "question_option_D_translated",
    "question_option_E_translated",
]

def letters_only_count(s: str) -> int:
    """Count alphabetic letters only (Unicode-aware: includes accents)."""
    return sum(1 for ch in s if ch.isalpha())

def chars_total_count(s: str) -> int:
    """Count total characters including spaces and punctuation."""
    return len(s)

# ---- Load data ----
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# ---- Compute counts ----
per_field_rows = []
per_question_rows = []

for item in data:
    year = str(item.get("year", "")).strip()
    field = str(item.get("field", "")).strip()
    qnum = item.get("question_number", None)

    question_key = f"{year}\t{field}_{qnum}"

    total_letters = 0
    total_chars = 0

    for fld in TEXT_FIELDS:
        text = item.get(fld, "")
        if not isinstance(text, str):
            text = ""  # be safe

        letters = letters_only_count(text)
        chars = chars_total_count(text)
        total_letters += letters
        total_chars += chars

        per_field_rows.append({
            "year": year,
            "field": field,
            "question_number": qnum,
            "question_key": question_key,
            "field_name": fld,
            "letters_only": letters,
            "chars_total": chars
        })

    per_question_rows.append({
        "year": year,
        "field": field,
        "question_number": qnum,
        "question_key": question_key,
        "letters_only_total": total_letters,
        "chars_total": total_chars
    })

# ---- Save CSVs ----
with open(per_field_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["year","field","question_number","question_key","field_name","letters_only","chars_total"]
    )
    writer.writeheader()
    writer.writerows(per_field_rows)

with open(per_question_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["year","field","question_number","question_key","letters_only_total","chars_total"]
    )
    writer.writeheader()
    writer.writerows(per_question_rows)

# ---- Quick console summary ----
print(f"Wrote per-field counts -> {per_field_csv}")
print(f"Wrote per-question totals -> {per_question_csv}\n")

# Show top 10 by letters_only_total
top10 = sorted(per_question_rows, key=lambda r: r["letters_only_total"], reverse=True)[:10]
print("Top 10 longest questions by LETTERS (sum across fields):")
for r in top10:
    print(f"  {r['question_key']}: {r['letters_only_total']} letters (chars={r['chars_total']})")


Wrote per-field counts -> C:\Users\thiago-ext\Documents\FNIRS\psychopy\question_letter_counts_per_field.csv
Wrote per-question totals -> C:\Users\thiago-ext\Documents\FNIRS\psychopy\question_letter_counts_totals.csv

Top 10 longest questions by LETTERS (sum across fields):
  2021	LC_26: 1124 letters (chars=1436)
  2020	LC_31: 1024 letters (chars=1245)
  2021	CH_83: 979 letters (chars=1219)
  2017	CH_75: 924 letters (chars=1137)
  2018	LC_44: 904 letters (chars=1088)
  2022	CH_60: 879 letters (chars=1065)
  2017	CH_47: 788 letters (chars=975)
  2022	CH_51: 779 letters (chars=945)
  2017	CH_69: 744 letters (chars=881)
  2022	CH_47: 744 letters (chars=923)


In [9]:
import csv
import statistics
from collections import Counter
import os
import math

# ====== CONFIG ======
file_path = r"C:\Users\thiago-ext\Documents\FNIRS\psychopy\question_letter_counts_totals.csv"

# Load totals
totals = []         # letters-only totals per question
question_keys = []  # optional: keep keys for reference
with open(file_path, "r", encoding="utf-8") as f:
    rdr = csv.DictReader(f)
    for row in rdr:
        try:
            totals.append(int(row["letters_only_total"]))
            question_keys.append(row["question_key"])
        except Exception:
            pass

if not totals:
    raise SystemExit("No data found. Make sure the CSV was generated and the path is correct.")

# Summary stats
mean_val = statistics.mean(totals)
median_val = statistics.median(totals)
min_val, max_val = min(totals), max(totals)

print(f"Questions counted: {len(totals)}")
print(f"Mean letters/question:   {mean_val:.2f}")
print(f"Median letters/question: {median_val:.2f}")
print(f"Min / Max:               {min_val} / {max_val}")

# Exact value distribution (value counts)
value_counts = Counter(totals)
# Show top 10 most common exact totals
print("\nTop 10 most common EXACT totals (letters):")
for val, cnt in value_counts.most_common(10):
    print(f"  {val:>6} letters : {cnt} question(s)")

# ---- Binned distribution (nice for a quick “histogram” in text) ----
# Choose a bin width; adjust as you like (e.g., 100, 200, 250)
BIN_WIDTH = 250

def to_bin(x, width):
    start = (x // width) * width
    end = start + width - 1
    return f"{start:>4}-{end:<4}"

binned = Counter(to_bin(x, BIN_WIDTH) for x in totals)

print(f"\nBinned distribution (bin width = {BIN_WIDTH} letters):")
for rng in sorted(binned, key=lambda r: int(r.split('-')[0])):
    print(f"  {rng} : {binned[rng]} question(s)")


Questions counted: 30
Mean letters/question:   711.10
Median letters/question: 698.00
Min / Max:               440 / 1124

Top 10 most common EXACT totals (letters):
     698 letters : 3 question(s)
     744 letters : 2 question(s)
     904 letters : 1 question(s)
    1024 letters : 1 question(s)
    1124 letters : 1 question(s)
     788 letters : 1 question(s)
     924 letters : 1 question(s)
     524 letters : 1 question(s)
     701 letters : 1 question(s)
     730 letters : 1 question(s)

Binned distribution (bin width = 250 letters):
   250-499  : 3 question(s)
   500-749  : 19 question(s)
   750-999  : 6 question(s)
  1000-1249 : 2 question(s)


In [10]:
import json
import re

# ========= CONFIG =========
file_path = r"C:\Users\thiago-ext\Documents\FNIRS\psychopy\filtered_questions_revised.json"
REMOVE = False  # set to True to strip detected citation lines and overwrite the JSON

# ========= Detection rules =========
# We treat as "citation/source" lines if they match ANY of these patterns.
# Tweak as needed.
URL_RE = re.compile(r'(https?://\S+|www\.\S+)', re.IGNORECASE)
AVAILABLE_AT_RE = re.compile(r'\bAvailable at\b', re.IGNORECASE)
ACCESSED_ON_RE = re.compile(r'\bAccessed on\b|\bAccessed:\b', re.IGNORECASE)
ADAPTED_RE = re.compile(r'\(adapted\)', re.IGNORECASE)

# Lines that look like references: AUTHOR, I.  Title. City: Publisher, 2006.
# (we keep it broad to catch “SILVEIRA, R.”, “TOCQUEVILLE, A.”, etc.)
AUTHOR_LINE_RE = re.compile(
    r'^[A-ZÁÉÍÓÚÂÊÔÃÕÇ][A-ZÁÉÍÓÚÂÊÔÃÕÇ\-\s]+,\s*[A-Z](?:\.[A-Z]\.)?[^0-9]*$'
)

# Lines with city/publisher/year patterns: "Rio de Janeiro: Zahar, 1979."
CITY_PUB_YEAR_RE = re.compile(
    r':\s*[^,]+,\s*(\d{4})\.?$'
)

# Lines starting with institutional names (UNESCO., ARISTOTLE, ARENDT, etc.) ending with year or (adapted)
INSTITUTION_LINE_RE = re.compile(
    r'^[A-Z][A-Za-z\.\s\-&]+,\s?.*\d{4}\.?(\s*\(adapted\))?$', re.IGNORECASE
)

def is_citation_line(line: str) -> bool:
    s = line.strip()
    if not s:
        return False
    # Quick contains checks
    if URL_RE.search(s): return True
    if AVAILABLE_AT_RE.search(s): return True
    if ACCESSED_ON_RE.search(s): return True
    if ADAPTED_RE.search(s): return True
    # Reference-style lines
    if AUTHOR_LINE_RE.search(s): return True
    if CITY_PUB_YEAR_RE.search(s): return True
    if INSTITUTION_LINE_RE.search(s): return True
    # Lines that are JUST an author/title chunk with no verbs and a year somewhere
    if re.search(r'\b(19|20)\d{2}\b', s) and (',' in s or ':' in s):
        # often a telltale of reference strings
        return True
    return False

# ========= Load JSON =========
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

total_q = 0
q_with_citations = 0
total_citation_lines = 0

report = []  # collect for nicer print at end

for item in data:
    total_q += 1
    year = str(item.get("year", "")).strip()
    field = str(item.get("field", "")).strip()
    qnum = item.get("question_number", "")
    key = f"{year}\t{field}_{qnum}"

    text = item.get("question_text_translated", "")
    if not isinstance(text, str) or not text.strip():
        continue

    lines = text.splitlines()
    citations_idx = [i for i, ln in enumerate(lines) if is_citation_line(ln)]

    if citations_idx:
        q_with_citations += 1
        total_citation_lines += len(citations_idx)

        # Build a snippet of matched lines
        matched = [f"    L{idx+1}: {lines[idx].strip()}" for idx in citations_idx]
        report.append(f"[{key}] citations detected:\n" + "\n".join(matched))

        if REMOVE:
            kept = [ln for i, ln in enumerate(lines) if i not in citations_idx]
            item["question_text_translated"] = "\n".join(kept)

# ========= Output =========
print(f"Scanned questions: {total_q}")
print(f"Questions with citation/source lines: {q_with_citations}")
print(f"Total citation/source lines detected: {total_citation_lines}\n")

if report:
    print("=== DETAILS ===")
    for block in report:
        print(block)
        print("-" * 60)
else:
    print("No citation/source lines detected by current rules.")

# ========= Save (optional) =========
if REMOVE:
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"\nCitations removed and file overwritten:\n{file_path}")
else:
    print("\n(Preview-only mode: set REMOVE = True to strip those lines and overwrite the JSON.)")


Scanned questions: 30
Questions with citation/source lines: 28
Total citation/source lines detected: 33

=== DETAILS ===
[2021	LC_26] citations detected:
    L3: For example, you probably don't know that the Rio de Janeiro author, who died in 1908, wrote lyrics for the national anthem in 1867 — and you couldn't really know, because the verses were still unpublished. Until today.
------------------------------------------------------------
[2017	CH_47] citations detected:
    L1: After the Universal Declaration of Human Rights by the UN in 1948, UNESCO published studies by scientists from around the world that disqualified racist doctrines and demonstrated the unity of the human race. Since then, most European scientists themselves have come to recognize the discriminatory nature of the white man's alleged racial superiority and to condemn the aberrations committed in his name.
    L2: SILVEIRA, R. The savages and the masses: role of scientific racism in the creation of Western hegemony

In [11]:
import json

file_path = r"C:\Users\thiago-ext\Documents\FNIRS\psychopy\filtered_questions_revised.json"

# Map of (year, field, question_number) -> list of EXACT citation lines to remove.
# We compare using .strip() so leading/trailing spaces/newlines don't matter.
CITATIONS_TO_REMOVE = {
    ("2017","CH",47): [
        "SILVEIRA, R. The savages and the masses: role of scientific racism in the creation of Western hegemony. Afro-Asia, n. 23, 1999 (adapted).",
    ],
    ("2017","CH",68): [
        "HAESBAERT, R. “Gauchos” and Bahians in the “new” Northeast: between economic globalization and the reinvention of territorial identities. In: CASTRO, I.E; GOMES, P.C.; CORRÊA, R.L. (Org.). Brazil: current issues of territorial reorganization. Rio de Janeiro: Bertrand Brasil, 2008.",
    ],
    ("2017","CH",69): [
        "Myrtle, Energy: the addiction of civilization, energy crisis and sustainable alternatives. Rio de Janeiro Caramond 2011",
    ],
    ("2017","CH",75): [
        "Palestinians celebrate increased status at the UN with flags and fireworks. Available at: http://folha.com. Accessed on: 4 Dec. 2012 (adapted).",
    ],
    ("2018","CH",77): [
        "WEIBEL, L. Available at: http://biblioteca.ibge.gov.br. Accessed on: 8 July 2015 (adapted).",
    ],
    ("2019","CH",56): [
        "SARTI, F.; HIRATUKA, C. World industry: recent changes and trends. Campinas: Unicamp, n. 186, Dec. 2010.",
    ],
    ("2019","CH",64): [
        "SILVA, C. C.; MARTINS, R A. Studies in the history and philosophy of science. São Paulo: Livraria da Physics, 2006 (adapted).",
    ],
    ("2019","CH",65): [
        "ARANHA, M. L. Machiavelli: the logic of force. São Paulo: Moderna, 2006 (adapted).",
    ],
    ("2020","CH",56): [
        "TUNDISI, J. G. Water resources in the future: problems and solutions. Advanced Studies, n. 63, 2008 (adapted).",
    ],
    ("2020","CH",59): [
        "Available at: http://ambientes.ambientebrasil.com.br. Accessed on: 25 June. 2015.",
    ],
    ("2020","CH",70): [
        "SALUSTIO. The Conjuration of Catiline/The War of Jugurtha. Petrópolis: Vozes, 1990 (adapted).",
    ],
    ("2020","CH",76): [
        "Belo Horizonte; São Paulo: Edusp, 1977. Adapted.)",
    ],
    ("2020","CH",77): [
        "ARISTOTLE Politics. Brasília: UnB, 1988.",
    ],
    ("2020","CH",78): [
        "(ANTUNES, R. The meanings of work: essay on the affirmation and denial of work. São Paulo: Boitempo. 2009. Adapted.)",
    ],
    ("2020","CH",82): [
        "ANGELO, C. Available at http://super.abril.com.br. Accessed on: 24 Oct. 2015 (adapted).",
    ],
    ("2020","CH",84): [
        "TEIXEIRA, W. et al. (Org.). Deciphering the Earth. São Paulo: Oficina de Textos, 2000.",
    ],
    ("2021","CH",46): [
        "Simply wanting to check messages from work after work takes a toll on your health - and that of your family. Available at: www.bbc.com. Accessed on: 4 Dec. 2018.",
    ],
    ("2021","CH",51): [
        "Water resources. Water for a sustainable world. UNESCO, 2015.",
    ],
    ("2021","CH",57): [
        "MARCUSE, H. The ideology of industrial society: the one-dimensional man.",
        "Rio de Janeiro: Zahar, 1979.",
    ],
    ("2021","CH",65): [
        "ENGELS, F. The situation of the working class in England. São Paulo: Boitempo, 2010.",
    ],
    ("2021","CH",76): [
        "Lisbon: Estampa, 1995.",
    ],
    ("2021","CH",83): [
        "RODRIGUES, M. R. A. M; TAVARES, A. C. P; Museological singularities of a board with sculptures in dialogue: from the alambamento to the wedding in Cabinda (Angola). Anais do Museu Paulista, n.2, May-Aug. 2017 (adapted).",
    ],
    ("2021","CH",86): [
        "DESCARTES, R. Principles of philosophy. Lisbon: Edições 70, 1997 (adapted).",
    ],
    ("2022","CH",47): [
        "ARENDT, H. The human condition. Rio de Janeiro: Forense Universitária, 2004.",
    ],
    ("2022","CH",51): [
        "CASTELLS, M. The network society – the information age:",
        "economy, society and culture. São Paulo: Peace and Land, 1999",
        "(adapted).",
    ],
    ("2022","CH",60): [
        "MOROZOV, E. Big Tech: the rise of data and the death of politics. São Paulo: Ubu, 2018 (adapted).",
    ],
    ("2022","CH",71): [
        "CANOTILHO, J. J. G. Rule of law, Lisbon: Gradiva, 1999 (adapted).",
    ],
    # NOTE: Skipping [2021, LC, 26] L3 (false positive narrative line).
}

def key_of(item):
    return (str(item.get("year","")).strip(),
            str(item.get("field","")).strip(),
            int(item.get("question_number", -1)))

def strip_citation_lines(text, lines_to_remove):
    """Remove only lines that match (after .strip()) any entry in lines_to_remove (also strip-compared)."""
    if not isinstance(text, str):
        return text, 0
    rm_set = {ln.strip() for ln in lines_to_remove}
    out_lines = []
    removed = 0
    for ln in text.splitlines():
        if ln.strip() in rm_set:
            removed += 1
        else:
            out_lines.append(ln)
    # tidy up extra blank lines (optional)
    new_text = "\n".join(out_lines).strip()
    return new_text, removed

# Load
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

touched = 0
removed_total = 0

for item in data:
    k = key_of(item)
    if k in CITATIONS_TO_REMOVE:
        before = item.get("question_text_translated", "")
        after, removed = strip_citation_lines(before, CITATIONS_TO_REMOVE[k])
        if removed > 0:
            item["question_text_translated"] = after
            touched += 1
            removed_total += removed
            print(f"Removed {removed} line(s) from {k[0]}\t{k[1]}_{k[2]}")
        # if nothing matched, we leave the text as-is silently

# Save
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"\nDone. Questions modified: {touched}")
print(f"Total citation lines removed: {removed_total}")


Removed 1 line(s) from 2017	CH_47
Removed 1 line(s) from 2017	CH_68
Removed 1 line(s) from 2017	CH_69
Removed 1 line(s) from 2017	CH_75
Removed 1 line(s) from 2018	CH_77
Removed 1 line(s) from 2019	CH_56
Removed 1 line(s) from 2019	CH_64
Removed 1 line(s) from 2019	CH_65
Removed 1 line(s) from 2020	CH_56
Removed 1 line(s) from 2020	CH_59
Removed 1 line(s) from 2020	CH_70
Removed 1 line(s) from 2020	CH_76
Removed 1 line(s) from 2020	CH_77
Removed 1 line(s) from 2020	CH_78
Removed 1 line(s) from 2020	CH_82
Removed 1 line(s) from 2020	CH_84
Removed 1 line(s) from 2021	CH_46
Removed 1 line(s) from 2021	CH_51
Removed 2 line(s) from 2021	CH_57
Removed 1 line(s) from 2021	CH_65
Removed 1 line(s) from 2021	CH_76
Removed 1 line(s) from 2021	CH_83
Removed 1 line(s) from 2021	CH_86
Removed 1 line(s) from 2022	CH_47
Removed 3 line(s) from 2022	CH_51
Removed 1 line(s) from 2022	CH_60
Removed 1 line(s) from 2022	CH_71

Done. Questions modified: 27
Total citation lines removed: 30
